# Part B — Text language features and separability

**Project:** Scalable Fraud Detection in Banking Chatbot Conversations
**Owner:** Wenhao Zhou · EDA, text side (第一部分) · figures B1, B2, B3

**Inputs (read-only, produced by the data pipeline):**

| file | content |
| --- | --- |
| `data/processed/banking77_clean.parquet` | 13,083 Banking77 messages, 77 intents, `split` = train 10,003 / test 3,080 |
| `data/processed/dim_intent.parquet` | one row per intent with `journey_stage`, `fraud_type`, `risk_level` (22 fraud-relevant intents, 6 fraud types) |

**Outputs:** `figures/B_*.png` and the supporting tables in `eda/output/`.

**How to run:** open this notebook from the `eda/` folder and use *Restart & Run All*.
Each analysis cell is self-contained — it builds its own features and model, writes its
figure, and prints the numbers quoted in the markdown below it.

In [ ]:
import os
import pathlib
import sys

# Run from eda/: repair the working directory if the notebook was opened from the repo root.
if not pathlib.Path("b1_confusion.py").exists() and pathlib.Path("eda/b1_confusion.py").exists():
    os.chdir("eda")
os.environ.setdefault("MPLCONFIGDIR", str(pathlib.Path("../.cache/mpl").resolve()))
pathlib.Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, os.getcwd())

import pandas as pd
from _common import DATA_DIR, FIG_DIR, OUT_DIR, load_data

clean, dim = load_data()
print("messages:", clean.shape, "| intents:", dim.shape)
print("split:", clean.groupby("split").size().to_dict())
print("fraud-relevant intents:", int(dim["fraud_relevant"].sum()),
      "| risk levels:", dim["risk_level"].value_counts().to_dict())

## B1 — Which intents get confused, and where does the fraud signal leak?
`figures/B_confusion_heatmap.png`

**Question.** Which of the 77 intents does a text model confuse, and which of those mix-ups
actually cost the bank money?

**Method.** TF-IDF (1–2 grams, `min_df=2`) + multinomial logistic regression, fit on the
`train` split and scored on `test`. The 77×77 matrix is summarised four ways: (a) the full
matrix ordered by journey stage, (b) a detail view of the intents in the 20 most frequent
mix-ups, (c) those 20 pairs ranked with a 500-sample bootstrap interval and stability column,
(d) the fraud leakage they cause, i.e. where each fraud-typed test message is actually routed.

**Reading the pairs.** A pair is *dangerous* when the two intents sit on different sides of
the fraud boundary (one has a `fraud_type`, the other does not) or in different fraud teams:
that is a message the fraud desk should have seen and did not. A pair is *harmless* when both
intents feed the same queue, so the customer is at worst answered by the wrong script.

In [ ]:
"""B1 - Intent confusion heat map and where the fraud signal is lost.

TF-IDF + multinomial logistic regression, trained on the `train` split only and
scored on the `test` split.

Panels
  1. 77 x 77 view, intents ordered by journey_stage, so blocks on the diagonal are
     confusions inside one stage and off-block cells are cross-stage errors.
  2. Detail view of the intents that take part in the 20 most frequent mix-ups.
  3. Those 20 pairs, split into "dangerous" (fraud queue lost or split) and
     "harmless" (both intents feed the same queue).
  4. Fraud leakage: for every message whose true intent is fraud-relevant, where
     the prediction actually sends it - same fraud team, another fraud team, or
     an ordinary servicing queue. Rates carry Wilson 95% intervals, and the pair
     ranking is resampled (bootstrap) because several counts are only 3-5.
"""

from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score

from _common import DANGER_COLOR, OUT_DIR, apply_style, load_data, save_figure, wilson

TOP_PAIRS = 20
CLIP = 8  # colour scale upper end; the diagonal is far above it
LEAK_COLOR = {"same_team": "#95a5a6", "wrong_team": "#e67e22", "ordinary": "#c0392b"}
RISK_RANK = {"high": 0, "medium": 1, "low": 2}


def fit_classifier(clean: pd.DataFrame):
    train = clean[clean["split"] == "train"]
    test = clean[clean["split"] == "test"]
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2), sublinear_tf=True, min_df=2, strip_accents="unicode"
    )
    x_train = vectorizer.fit_transform(train["text"])
    x_test = vectorizer.transform(test["text"])
    model = LogisticRegression(max_iter=2000, C=10.0, random_state=42).fit(
        x_train, train["label"]
    )
    return train, test, model.predict(x_test), model


def label_pairs(cm: np.ndarray, names: list[str], top: int = TOP_PAIRS) -> pd.DataFrame:
    """Rank undirected intent pairs by how often the test set mixed them up."""
    rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            forward, backward = int(cm[i, j]), int(cm[j, i])
            if forward + backward == 0:
                continue
            rows.append({"i": i, "j": j, "a": names[i], "b": names[j],
                         "count": forward + backward, "a_to_b": forward, "b_to_a": backward})
    pairs = pd.DataFrame(rows).sort_values(["count", "i", "j"], ascending=[False, True, True])
    return pairs.head(top).reset_index(drop=True)


def bootstrap_pairs(test, pred, n_intents: int, n_boot: int = 500,
                    top: int = TOP_PAIRS) -> pd.DataFrame:
    """Resample the test set to see how stable the top-pair ranking is."""
    rng = np.random.default_rng(7)
    flat = test["label"].to_numpy() * n_intents + pred
    tri = np.triu_indices(n_intents, 1)
    n_pairs = len(tri[0])
    vals = np.empty((n_boot, n_pairs), dtype=np.int32)
    for b in range(n_boot):
        sample = flat[rng.integers(0, len(flat), len(flat))]
        cmb = np.bincount(sample, minlength=n_intents ** 2).reshape(n_intents, n_intents)
        vals[b] = cmb[tri] + cmb.T[tri]

    in_top = np.zeros(n_pairs)
    for b in range(n_boot):
        in_top[np.argsort(-vals[b])[:top]] += 1

    return pd.DataFrame({
        "i": tri[0], "j": tri[1],
        "stab": in_top / n_boot,
        "lo": np.percentile(vals, 2.5, axis=0),
        "hi": np.percentile(vals, 97.5, axis=0),
    })


def classify_pair(row, ftype: dict) -> str:
    fa, fb = ftype[row["a"]], ftype[row["b"]]
    if pd.isna(fa) and pd.isna(fb):
        return "harmless"
    if fa == fb:
        return "harmless"
    return "dangerous"


def fraud_leakage(cm: np.ndarray, dim: pd.DataFrame) -> pd.DataFrame:
    """For each true fraud type, where do its test messages end up?"""
    name = dim.sort_values("intent_id")["intent"].tolist()
    ftype = dict(zip(name, dim.sort_values("intent_id")["fraud_type"]))
    rows = []
    for i, true_name in enumerate(name):
        if pd.isna(ftype[true_name]):
            continue
        for j, pred_name in enumerate(name):
            n = int(cm[i, j])
            if n == 0:
                continue
            if ftype[true_name] == ftype[pred_name]:
                bucket = "same_team"
            elif pd.isna(ftype[pred_name]):
                bucket = "ordinary"
            else:
                bucket = "wrong_team"
            rows.append({"fraud_type": ftype[true_name], "true": true_name,
                         "pred": pred_name, "n": n, "bucket": bucket})
    detail = pd.DataFrame(rows)
    table = (
        detail.pivot_table(index="fraud_type", columns="bucket", values="n",
                           aggfunc="sum", fill_value=0)
        .reindex(columns=["same_team", "wrong_team", "ordinary"], fill_value=0)
    )
    table["n"] = table.sum(axis=1)
    table = table.sort_values("n", ascending=False)
    return table, detail


def draw(pairs, detail_cm, overview_cm, names, dim, metrics, leak_table, order, stage_names):
    ftype = dict(zip(names, dim.sort_values("intent_id")["fraud_type"]))
    risk = dict(zip(names, dim.sort_values("intent_id")["risk_level"]))
    off_diag = detail_cm.sum(axis=1) - np.diag(detail_cm)
    # Clip the colour scale well below the diagonal so off-diagonal structure shows.
    vmax = np.log1p(CLIP)

    fig = plt.figure(figsize=(23, 15.5))
    outer = fig.add_gridspec(1, 2, width_ratios=[1.06, 1.0], wspace=0.34,
                             left=0.035, right=0.985, top=0.925, bottom=0.055)
    left = outer[0].subgridspec(2, 1, height_ratios=[0.95, 1.0], hspace=0.20)
    right = outer[1].subgridspec(2, 1, height_ratios=[1.62, 1.0], hspace=0.34)
    ax_o, ax_h, ax_b, ax_l = (fig.add_subplot(s) for s in
                              (left[0], left[1], right[0], right[1]))

    # ---- 1. full 77 x 77 view, ordered by journey stage -------------------------
    plot_o = np.log1p(overview_cm[np.ix_(order, order)])
    im_o = ax_o.imshow(plot_o, cmap="Blues", vmin=0, vmax=vmax, aspect="equal",
                       interpolation="nearest")

    stage_pos = {}
    for pos, k in enumerate(order):
        stage_pos.setdefault(stage_names[k], []).append(pos)
    bounds = [0] + [max(v) + 0.5 for v in stage_pos.values()]
    for b in bounds[1:-1]:
        ax_o.axhline(b, color="white", lw=0.9)
        ax_o.axvline(b, color="white", lw=0.9)
    ax_o.set_yticks([np.mean(v) for v in stage_pos.values()])
    ax_o.set_yticklabels(list(stage_pos), fontsize=5.6)
    ax_o.set_xticks([])
    ax_o.set_xlabel("predicted intent (same stage order)", fontsize=8)
    ax_o.set_ylabel("true intent, grouped by journey stage", fontsize=8)
    ax_o.set_title("All 77 intents, ordered by journey stage\n"
                   "blocks on the diagonal = mix-ups inside one stage",
                   loc="left", fontsize=10)

    # ---- 2. detail view ---------------------------------------------------------
    involved = sorted({int(v) for v in pd.concat([pairs["i"], pairs["j"]])})
    if len(involved) > 26:
        involved = sorted(involved, key=lambda k: -off_diag[k])[:26]
    involved.sort(key=lambda k: (stage_names[k], RISK_RANK[risk[names[k]]], -off_diag[k]))

    sub = detail_cm[np.ix_(involved, involved)]
    short = [names[k] for k in involved]
    plot = np.log1p(sub)
    ax_h.imshow(plot, cmap="Blues", vmin=0, vmax=vmax, aspect="auto")
    ax_h.set_xticks(range(len(short)), short, rotation=90, fontsize=6.2)
    ax_h.set_yticks(range(len(short)), short, fontsize=6.2)
    for tick, k in zip(ax_h.get_yticklabels(), involved):
        tick.set_color({"high": "#c0392b", "medium": "#e67e22", "low": "#4d4d4d"}[risk[names[k]]])
    for r in range(len(short)):
        for c in range(len(short)):
            if sub[r, c] >= 2:
                ax_h.text(c, r, int(sub[r, c]), ha="center", va="center", fontsize=5.8,
                          color="white" if plot[r, c] >= vmax * 0.82 else "#333333")
    for _, row in pairs.iterrows():
        for a, b in ((row["i"], row["j"]), (row["j"], row["i"])):
            if a in involved and b in involved:
                r, c = involved.index(a), involved.index(b)
                ax_h.add_patch(Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False,
                                         edgecolor=DANGER_COLOR["dangerous"], lw=1.0, ls=":"))
    ax_h.set_xlabel("predicted intent")
    ax_h.set_ylabel("true intent")
    ax_h.set_title(f"Intents in the top {len(pairs)} mix-ups, ordered by stage then risk\n"
                   "cell = test messages; 1-message cells unlabelled; dotted = top pairs",
                   loc="left", fontsize=10)
    cb = fig.colorbar(im_o, ax=ax_o, fraction=0.030, pad=0.025)
    cb.set_label(f"log1p(test messages), clipped at {CLIP}", fontsize=7.5)

    # ---- 3. ranked pairs --------------------------------------------------------
    pairs = pairs.assign(kind=[classify_pair(r, ftype) for _, r in pairs.iterrows()])
    ordered = pairs.sort_values("count")
    xerr = np.vstack([ordered["count"] - ordered["lo"], ordered["hi"] - ordered["count"]]).clip(min=0)
    bars_b = ax_b.barh(range(len(ordered)), ordered["count"],
              color=[DANGER_COLOR[k] for k in ordered["kind"]], height=0.62,
              xerr=xerr, ecolor="#4d4d4d", capsize=2.2, error_kw={"lw": 0.8, "alpha": 0.85})
    ax_b.set_yticks(range(len(ordered)),
                    [f"{r.a}\n\u21c4  {r.b}" for r in ordered.itertuples()],
                    fontsize=6.9, linespacing=1.3)
    ax_b.set_xlabel("test messages confused across the pair (both directions)")
    n_d = int((pairs["kind"] == "dangerous").sum())
    ax_b.set_title(f"Top {len(pairs)} confused intent pairs: "
                   f"{n_d} dangerous (red), {len(pairs) - n_d} harmless (blue)\n"
                   "bar = test messages, whisker = 95% bootstrap interval, "
                   "right column = share of 500 resamples keeping the pair in the top 20",
                   loc="left", fontsize=9.5)
    # A separate right-hand column keeps the bootstrap stability readable.
    xmax = ordered["hi"].max()
    ax_b.set_xlim(0, xmax * 1.20)
    ax_b.axvline(xmax * 1.04, color="#b0b0b0", lw=0.7, ls=":")
    for yi, r in enumerate(ordered.itertuples()):
        ax_b.text(xmax * 1.17, yi, f"{r.stab:.0%}", va="center", ha="right",
                  fontsize=6.8, color="#555555")
    ax_b.grid(axis="x", alpha=0.25, lw=0.6)
    ax_b.set_axisbelow(True)
    ax_b.legend(handles=[Patch(color=DANGER_COLOR["dangerous"], label="dangerous"),
                         Patch(color=DANGER_COLOR["harmless"], label="harmless")],
                loc="upper left", bbox_to_anchor=(0.0, -0.055), ncol=2,
                frameon=False, fontsize=8)

    # ---- 4. fraud leakage -------------------------------------------------------
    order_l = list(leak_table.index)
    y = np.arange(len(order_l))
    left_cum = np.zeros(len(order_l))
    for bucket in ("same_team", "wrong_team", "ordinary"):
        vals = leak_table[bucket].to_numpy()
        ax_l.barh(y, vals, left=left_cum, height=0.66, color=LEAK_COLOR[bucket],
                  label=bucket.replace("_", " "))
        left_cum += vals
    ax_l.set_yticks(y, [t.replace("_", " ") for t in order_l], fontsize=8)
    ax_l.invert_yaxis()
    for yi, (idx, row) in enumerate(leak_table.iterrows()):
        ax_l.text(row["n"] + leak_table["n"].max() * 0.012, yi,
                  f"n={int(row['n'])}\n{row['p']:.1%} [{row['lo']:.0%}-{row['hi']:.0%}] to ordinary",
                  va="center", fontsize=7.2, linespacing=1.35,
                  color=LEAK_COLOR["ordinary"] if row["p"] > 0.08 else "#444444")
    total = int(leak_table["n"].sum())
    leak_all = int(leak_table["ordinary"].sum())
    wrong_all = int(leak_table["wrong_team"].sum())
    lo_all, hi_all = wilson(leak_all, total)
    ax_l.set_xlim(0, leak_table["n"].max() * 1.52)
    ax_l.set_xlabel("test messages whose true intent is fraud-relevant")
    ax_l.set_title(f"Fraud leakage: {leak_all} of {total} risky test messages "
                   f"({leak_all / total:.1%}, 95% CI {lo_all:.1%}-{hi_all:.1%}) leave the fraud workflow\n"
                   f"another {wrong_all} ({wrong_all / total:.1%}) land in a different fraud team; "
                   f"intervals are Wilson, n is small",
                   loc="left", fontsize=10)
    ax_l.legend(loc="lower right", frameon=False, fontsize=7.6, title="predicted queue",
                title_fontsize=7.6)
    ax_l.grid(axis="x", alpha=0.25, lw=0.6)
    ax_l.set_axisbelow(True)

    fig.suptitle(
        f"B1  Intent confusion: TF-IDF + logistic regression   |   "
        f"test accuracy {metrics['accuracy']:.3f}, macro-F1 {metrics['macro_f1']:.3f} "
        f"(77 intents, {metrics['n_test']} held-out messages)   |   "
        f"fraud leakage {leak_all / total:.1%} [{lo_all:.1%}-{hi_all:.1%}] of risky messages",
        x=0.012, ha="left", fontsize=14, fontweight="bold",
    )
    return fig, pairs


def main() -> None:
    apply_style()
    clean, dim = load_data()
    train, test, pred, model = fit_classifier(clean)

    dim = dim.sort_values("intent_id").reset_index(drop=True)
    names = dim["intent"].tolist()
    stage_names = dim["journey_stage"].tolist()

    cm = confusion_matrix(test["label"], pred, labels=list(range(len(names))))
    off_diag = cm.sum(axis=1) - np.diag(cm)
    stage_order = sorted(set(stage_names))
    order = sorted(
        range(len(names)),
        key=lambda k: (stage_order.index(stage_names[k]),
                       RISK_RANK[dim["risk_level"][k]], -off_diag[k]),
    )

    metrics = {
        "n_train": int(len(train)),
        "n_test": int(len(test)),
        "accuracy": float((pred == test["label"].to_numpy()).mean()),
        "macro_f1": float(f1_score(test["label"], pred, average="macro")),
        "weighted_f1": float(f1_score(test["label"], pred, average="weighted")),
        "n_features": int(model.n_features_in_),
    }
    pairs = label_pairs(cm, names)
    pairs = pairs.merge(bootstrap_pairs(test, pred, len(names)), on=["i", "j"], how="left")
    leak_table, leak_detail = fraud_leakage(cm, dim)
    leak_table["p"] = leak_table["ordinary"] / leak_table["n"]
    bounds_ci = [wilson(int(r.ordinary), int(r.n)) for r in leak_table.itertuples()]
    leak_table["lo"] = [b[0] for b in bounds_ci]
    leak_table["hi"] = [b[1] for b in bounds_ci]

    fig, pairs = draw(pairs, cm, cm, names, dim, metrics, leak_table, order, stage_names)
    save_figure(fig, "B_confusion_heatmap.png")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pairs.to_csv(OUT_DIR / "b1_top_pairs.csv", index=False)
    leak_table.to_csv(OUT_DIR / "b1_leakage_by_fraud_type.csv")
    leak_detail.sort_values("n", ascending=False).to_csv(
        OUT_DIR / "b1_leakage_detail.csv", index=False)
    (OUT_DIR / "b1_metrics.json").write_text(json.dumps(metrics, indent=2))

    print(json.dumps(metrics, indent=2))
    print(pairs[["a", "b", "count", "lo", "hi", "stab", "kind"]].to_string(index=False))
    print("\nleakage by fraud type:\n", leak_table.to_string())
    total = leak_table["n"].sum()
    print(f"\nrisky test messages {total}, to ordinary {leak_table['ordinary'].sum()} "
          f"({leak_table['ordinary'].sum() / total:.1%}), "
          f"wrong fraud team {leak_table['wrong_team'].sum()} "
          f"({leak_table['wrong_team'].sum() / total:.1%})")


main()

In [ ]:
from IPython.display import Image, display
from _common import FIG_DIR, OUT_DIR
import pandas as pd

display(Image(filename=str(FIG_DIR / "B_confusion_heatmap.png")))
pd.read_csv(OUT_DIR / "b1_top_pairs.csv")[["a", "b", "count", "lo", "hi", "stab", "kind"]]

**Interpretation (3 sentences).** TF-IDF + logistic regression classifies the 77 intents at
0.891 accuracy on the 3,080 held-out messages, and most mistakes stay inside one journey
stage; of the 20 most frequent mix-ups, 15 are harmless because both intents feed the same
desk, while 5 push a card-compromise or money-mule message into a routine servicing queue.
Aggregated over the 880 fraud-typed test messages, 56 (6.4%, 95% CI 4.9–8.2%) never reach a
fraud workflow at all, and virtual-card abuse leaks 12.5% — for a bank that is a chargeback
and a complaint, not just a mislabelled row. The sharpest text-only failure is the PIN
cluster, where "How do I get my PIN unlocked?" (`pin_blocked`, account takeover) and
"I need my PIN" (`get_physical_card`, card logistics) share almost every word, so text alone
cannot route them.

**Caveats.** Only the first six pairs survive in >80% of bootstrap resamples, and three of the
five "dangerous" pairs recur in under half, so they are suggestive patterns rather than a
stable ranking.

## B2 — What words do each fraud type's customers actually use?
`figures/B_keywords_by_fraud_type.png`

**Question.** Per fraud type, which words and phrases are characteristic, and are they enough
to detect that type on their own?

**Method.** Log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi & Quinn,
"Fightin' Words"): each fraud type is compared with the rest of the **train** messages, so a
term scores high only when that type over-uses it. Terms are unigrams and bigrams, greedily
de-duplicated so "virtual card" cannot push both "virtual" and "card" out of the list. The
bottom panel takes the same top-10 terms as a standalone detector — a message takes the
best-scoring type — and scores it on the held-out `test` split.

In [ ]:
"""B2 - Words and phrases that mark each fraud type, and how far they carry.

Top panels: log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi &
Quinn, "Fightin' Words", 2008). Each fraud type is compared with the rest of the
training messages, so a term scores high only when that type over-uses it.

    delta_w = log((y_aw + a_w) / (n_a + a - y_aw - a_w))
            - log((y_bw + a_w) / (n_b + a - y_bw - a_w))
    var_w   = 1/(y_aw + a_w) + 1/(n_a + a - y_aw - a_w)
            + 1/(y_bw + a_w) + 1/(n_b + a - y_bw - a_w)
    z_w     = delta_w / sqrt(var_w)

Terms are unigrams and bigrams, greedily de-duplicated so "virtual card" cannot
push both "virtual" and "card" out of the list.

Bottom panel: can those terms act as a fraud rule on their own? The vocabulary and
the term lists are built on the `train` split only; the rule (a message is scored
for a type by the number of its terms it contains, and takes the best-scoring type)
is then run on the `test` split. Recall and precision are reported per fraud type.
"""

from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer

from _common import OUT_DIR, RISK_COLOR, apply_style, load_data, save_figure

PRIOR = 0.5
TOP_WORDS = 10
MIN_DF = 5

# Drop function words but keep negations: "not recognised" and "didn't" carry the fraud signal.
KEEP = {"not", "no", "nor", "never", "cannot", "none", "nothing",
        "didn't", "don't", "doesn't", "isn't", "wasn't", "aren't", "weren't",
        "can't", "couldn't", "won't", "wouldn't", "shouldn't", "haven't",
        "hasn't", "hadn't", "unable", "without"}
# A few auxiliaries missing from sklearn's list would otherwise form junk bigrams ("did not").
EXTRA_STOP = {"did", "does", "having", "had", "been", "being"}
STOP_WORDS = sorted((set(ENGLISH_STOP_WORDS) - KEEP) | EXTRA_STOP)


def build_counts(clean: pd.DataFrame, dim: pd.DataFrame):
    """Vocabulary and counts fit on train only, then applied to every row."""
    text = clean["text"].str.replace("recognize", "recognise", regex=False)
    train_mask = (clean["split"] == "train").to_numpy()
    vectorizer = CountVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-z][a-z'\-]+\b",
        min_df=MIN_DF,
        stop_words=STOP_WORDS,
        strip_accents="unicode",
    )
    vectorizer.fit(text[train_mask])
    matrix = vectorizer.transform(text)
    vocab = np.array(vectorizer.get_feature_names_out())
    ftype = dict(zip(dim["intent"], dim["fraud_type"]))
    doc_type = clean["intent"].map(ftype)
    return matrix, vocab, doc_type, train_mask


def fightin_words(y_a, n_a, y_b, n_b):
    total = y_a + y_b
    prior = PRIOR * total / (n_a + n_b)
    p_a = (y_a + prior) / (n_a + PRIOR - y_a - prior)
    p_b = (y_b + prior) / (n_b + PRIOR - y_b - prior)
    delta = np.log(p_a) - np.log(p_b)
    var = (1.0 / (y_a + prior) + 1.0 / (n_a + PRIOR - y_a - prior)
           + 1.0 / (y_b + prior) + 1.0 / (n_b + PRIOR - y_b - prior))
    return delta, delta / np.sqrt(var)


def top_words(matrix, vocab, doc_type, fraud_type, train_mask, top=TOP_WORDS) -> pd.DataFrame:
    counts = np.asarray(matrix[train_mask].sum(axis=0)).ravel()
    in_group = (doc_type == fraud_type).to_numpy() & train_mask
    y_a = np.asarray(matrix[in_group].sum(axis=0)).ravel()
    y_b = counts - y_a
    n_a, n_b = int(y_a.sum()), int(y_b.sum())

    delta, z = fightin_words(y_a, n_a, y_b, n_b)
    table = pd.DataFrame({"word": vocab, "count_in_type": y_a.astype(int),
                          "count_elsewhere": y_b.astype(int),
                          "delta": delta, "z": z})
    table["rate_in_type"] = table["count_in_type"] / n_a * 1000
    table["rate_elsewhere"] = table["count_elsewhere"] / n_b * 1000
    table = table.sort_values("z", ascending=False)

    # Greedy de-duplication: a selected term blocks later terms sharing a token.
    chosen, used = [], set()
    for _, row in table.iterrows():
        tokens = set(str(row["word"]).split())
        if tokens & used:
            continue
        chosen.append(row)
        used |= tokens
        if len(chosen) == top:
            break
    return pd.DataFrame(chosen).reset_index(drop=True)


def rule_eval(matrix, vocab, doc_type, results, train_mask) -> pd.DataFrame:
    """Score every test message with each type's term list; take the best-scoring type."""
    test_idx = np.where(~train_mask)[0]
    col = {w: i for i, w in enumerate(vocab)}
    types = list(results)

    score = np.zeros((len(test_idx), len(types)))
    for t, fraud_type in enumerate(types):
        cols = [col[w] for w in results[fraud_type]["word"] if w in col]
        hits = np.asarray((matrix[test_idx][:, cols] > 0).sum(axis=1)).ravel()
        score[:, t] = hits

    best = score.argmax(axis=1)
    pred = np.array(types, dtype=object)[best]
    pred[score.max(axis=1) == 0] = None
    true = doc_type.iloc[test_idx].to_numpy()

    rows = []
    for fraud_type in types:
        is_true = true == fraud_type
        is_pred = pred == fraud_type
        tp = int((is_true & is_pred).sum())
        rows.append({
            "fraud_type": fraud_type,
            "n_true": int(is_true.sum()),
            "n_pred": int(is_pred.sum()),
            "tp": tp,
            "recall": tp / max(int(is_true.sum()), 1),
            "precision": tp / max(int(is_pred.sum()), 1),
        })
    table = pd.DataFrame(rows).set_index("fraud_type")
    table.loc["ALL"] = {
        "n_true": int(pd.notna(true).sum()),
        "n_pred": int(pd.notna(pred).sum()),
        "tp": int((pd.notna(true) & (pred == true)).sum()),
        "recall": float((pred == true)[pd.notna(true)].mean()),
        "precision": float((pred == true)[pd.notna(pred)].mean()),
    }
    return table


def draw(results, sizes, dim, rule):
    risk = (dim.dropna(subset=["fraud_type"]).drop_duplicates("fraud_type")
            .set_index("fraud_type")["risk_level"].to_dict())
    order = sorted(results, key=lambda f: (-sizes[f], f))

    fig = plt.figure(figsize=(17.5, 13.2))
    grid = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.72], hspace=0.42, wspace=0.26,
                            left=0.055, right=0.985, top=0.90, bottom=0.075)
    axes = [fig.add_subplot(grid[r, c]) for r in range(2) for c in range(3)]
    ax_rule = fig.add_subplot(grid[2, :])

    for ax, fraud_type in zip(axes, order):
        table = results[fraud_type].iloc[::-1]
        color = RISK_COLOR[risk[fraud_type]]
        ax.barh(range(len(table)), table["z"], color=color, height=0.72, alpha=0.9)
        ax.set_yticks(range(len(table)), table["word"], fontsize=9)
        ax.set_xlim(0, max(table["z"].max() * 1.18, 2.4))
        ax.bar_label(ax.containers[0], labels=[f"{z:.0f}" for z in table["z"]],
                     padding=2, fontsize=7.4, color="#444444")
        ax.axvline(1.96, color="#888888", lw=0.8, ls=":")
        ax.set_title(f"{fraud_type.replace('_', ' ')}  "
                     f"({risk[fraud_type]} risk, {sizes[fraud_type]:,} train messages)",
                     loc="left", fontsize=10.5)
        ax.grid(axis="x", alpha=0.25, lw=0.6)
        ax.set_axisbelow(True)
        ax.spines["left"].set_visible(False)

    # ---- rule evaluation --------------------------------------------------------
    bars = rule.drop(index="ALL").sort_values("recall", ascending=False)
    y = np.arange(len(bars))
    ax_rule.barh(y - 0.19, bars["recall"], height=0.36, color="#2c6e9b",
                 label="recall (share of that type caught)")
    ax_rule.barh(y + 0.19, bars["precision"], height=0.36, color="#8fb8d0",
                 label="precision (share of flags that are that type)")
    ax_rule.set_yticks(y, [t.replace("_", " ") for t in bars.index], fontsize=9)
    ax_rule.invert_yaxis()
    ax_rule.set_xlim(0, 1.06)
    ax_rule.set_xlabel("share of test messages")
    for yi, (idx, row) in enumerate(bars.iterrows()):
        ax_rule.text(row["recall"] + 0.012, yi - 0.19, f"{row['recall']:.0%}",
                     va="center", fontsize=7.6, color="#2c6e9b")
        ax_rule.text(row["precision"] + 0.012, yi + 0.19, f"{row['precision']:.0%}",
                     va="center", fontsize=7.6, color="#5b7f95")
    all_row = rule.loc["ALL"]
    ax_rule.set_title(
        "If the top-10 terms per type were the whole detector (terms fit on train, "
        f"scored on test): overall recall {all_row['recall']:.0%}, "
        f"precision {all_row['precision']:.0%}, "
        f"{int(all_row['n_pred'])} test messages flagged",
        loc="left", fontsize=10,
    )
    ax_rule.legend(loc="lower right", frameon=False, fontsize=8.5)
    ax_rule.grid(axis="x", alpha=0.25, lw=0.6)
    ax_rule.set_axisbelow(True)

    fig.suptitle(
        "B2  Words and phrases that mark each fraud type (log-odds ratio with Dirichlet prior)\n"
        "Top: each type against the rest of the train messages; dotted line = z 1.96 (~95%). "
        "Bottom: the same terms used as a standalone rule on test",
        x=0.012, y=0.985, ha="left", fontsize=13, fontweight="bold",
    )
    fig.legend(handles=[Patch(color=RISK_COLOR["high"], label="high risk fraud type"),
                        Patch(color=RISK_COLOR["medium"], label="medium risk fraud type")],
               loc="upper right", bbox_to_anchor=(0.995, 0.945), ncol=2,
               frameon=False, fontsize=9)
    fig.text(0.5, 0.012,
             "log-odds z-score of each term in that fraud type against the rest of the "
             "train messages; the x scale differs by panel",
             ha="center", fontsize=8.8, color="#444444")
    return fig


def main() -> None:
    apply_style()
    clean, dim = load_data()
    matrix, vocab, doc_type, train_mask = build_counts(clean, dim)
    sizes = doc_type[train_mask].dropna().value_counts().to_dict()

    results = {ft: top_words(matrix, vocab, doc_type, ft, train_mask) for ft in sizes}
    rule = rule_eval(matrix, vocab, doc_type, results, train_mask)

    fig = draw(results, sizes, dim, rule)
    save_figure(fig, "B_keywords_by_fraud_type.png")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.concat([t.assign(fraud_type=ft) for ft, t in results.items()],
              ignore_index=True).to_csv(OUT_DIR / "b2_top_words.csv", index=False)
    rule.to_csv(OUT_DIR / "b2_keyword_rule_eval.csv")
    (OUT_DIR / "b2_group_sizes.json").write_text(
        json.dumps({k: int(v) for k, v in sorted(sizes.items())}, indent=2))

    print("train group sizes:", {k: int(v) for k, v in sorted(sizes.items())})
    for ft, table in results.items():
        print(f"\n{ft}")
        print(table[["word", "rate_in_type", "rate_elsewhere", "z"]]
              .to_string(index=False, float_format=lambda v: f"{v:0.2f}"))
    print("\nkeyword-rule evaluation on test:\n", rule.round(3).to_string())


main()

In [ ]:
from IPython.display import Image, display
from _common import FIG_DIR, OUT_DIR
import pandas as pd

display(Image(filename=str(FIG_DIR / "B_keywords_by_fraud_type.png")))
pd.read_csv(OUT_DIR / "b2_keyword_rule_eval.csv", index_col=0).round(3)

**Interpretation (3 sentences).** Every fraud type has its own vocabulary — `pin`, `change`,
`details` for account takeover; `transfer`, `cancel`, `beneficiary` for money mules;
`recognise`, `withdrawal`, `stolen` for card compromise — and these are ordinary
account-journey nouns and verbs rather than threats, so a bank should build one keyword layer
per fraud typology instead of a single generic "fraud lexicon". Used as a standalone rule the
top-10 terms recover 77% of fraud-typed test messages, but only 29% of the flags are correct,
which for a bank means roughly 1,700 innocent customers sent into a fraud review to catch 678
real cases. `identity_kyc_bypass` is the only type where the rule nearly works (84% recall,
63% precision), while `card_compromise` is right just 14% of the time because an unrecognised
payment reads exactly like an ordinary card complaint.

**Caveats.** A term being discriminative does not mean it is causal or stable: these are
intent-level words, so a fraud detector built on them mostly relearns the 77-way intent
classifier — which is why the fraud model has to add behaviour on top of text.

## B3 — Does urgency language separate high-risk messages?
`figures/B_urgency_by_risk.png`

**Question.** Do high-risk intents "sound" more urgent or emotional than ordinary ones?

**Method.** A hand-built lexicon from the brief in three groups — time pressure
(`immediately`, `urgent`, `asap`), distress (`help`, `worried`, `scared`) and incident
(`someone`, `stolen`, `fraud`) — flags each message once. The figure reports (a) the share
carrying a term by risk level with Wilson intervals and a chi-square test, (b) the same by
fraud type against an ordinary reference, (c) the odds ratio high/medium vs low with and
without log message length in the model (bootstrap intervals), and (d) the per-intent spread,
so the gap between risk levels can be compared with the gap inside one level. A lexicon
sensitivity table (brief-only, strict, current, broad) is written to `eda/output/`.

In [ ]:
"""B3 - Does urgency language separate high-risk messages from ordinary ones?

A hand-built lexicon in three groups (time pressure, distress/appeal, incident/threat)
flags each Banking77 message once. The figure asks four things:

  a. What share of messages carries each group, by risk level? Wilson 95% intervals,
     plus a chi-square test across the three levels.
  b. Which fraud types produce the effect? Fraud types against an ordinary reference.
  c. Is the gap an artefact of message length? A message that says more has more
     chances to contain a keyword, so the panel compares the odds ratio between risk
     levels with and without log message length in the model (bootstrap intervals).
  d. Is the risk label a good proxy at all? Each dot is one of the 77 intents, so the
     spread inside a risk level can be compared with the gap between levels.

A lexicon-sensitivity table (brief-only terms, strict urgency, current, broad) is
written to CSV: the conclusions must not depend on how the word list was drawn.
"""

from __future__ import annotations

import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FixedLocator, NullFormatter
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression

from _common import OUT_DIR, RISK_COLOR, apply_style, load_data, save_figure, wilson

LEXICON = {
    "time pressure": [
        "immediately", "urgent", "asap", "right away", "as soon as possible", "hurry",
        "quickly", "emergency", "today", "straight away", "instantly", "now",
    ],
    "distress / appeal": [
        "help", "please", "worried", "worry", "scared", "panic", "desperate", "upset",
        "angry", "frustrated", "afraid", "stressed", "terrible", "awful",
    ],
    "incident / threat": [
        "someone", "stolen", "steal", "fraud", "scam", "hacked", "unauthorised",
        "unauthorized", "suspicious", "fake", "compromised", "not mine",
        "without my permission",
    ],
}
CAT_COLOR = {"time pressure": "#e08a1e", "distress / appeal": "#8e6bbf",
             "incident / threat": "#c0392b"}
RISK_LABEL = {"low": "low\n(55 intents)", "medium": "medium\n(10)", "high": "high\n(12)"}

ALL_TERMS = [w for words in LEXICON.values() for w in words]
SENSITIVITY = {
    "brief only": ["immediately", "urgent", "someone", "stolen", "help"],
    "strict urgency": ["immediately", "urgent", "asap", "emergency", "straight away",
                       "right away", "as soon as possible", "hurry", "scared", "panic",
                       "desperate"],
    "current (3 groups)": ALL_TERMS,
    "broad (+plea)": ALL_TERMS + ["need", "sorry", "problem", "wrong"],
}


def _pattern(words: list[str]) -> re.Pattern:
    joined = "|".join(re.escape(w) for w in sorted(words, key=len, reverse=True))
    return re.compile(rf"\b(?:{joined})\b", re.IGNORECASE)


def build_flags(clean: pd.DataFrame, dim: pd.DataFrame) -> pd.DataFrame:
    frame = clean.merge(dim[["intent", "risk_level", "fraud_type"]], on="intent", how="left")
    for name, words in LEXICON.items():
        frame[name] = frame["text"].str.contains(_pattern(words), regex=True)
    frame["any term"] = frame[list(LEXICON)].any(axis=1)
    return frame


def share_table(frame: pd.DataFrame, by: str) -> pd.DataFrame:
    rows = []
    for key, group in frame.groupby(by, observed=True):
        for cat in list(LEXICON) + ["any term"]:
            k, n = int(group[cat].sum()), int(len(group))
            lo, hi = wilson(k, n)
            rows.append({by: key, "group": cat, "k": k, "n": n,
                         "share": k / n, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)


def chi_square(frame: pd.DataFrame, cat: str) -> tuple[float, float]:
    table = pd.crosstab(frame["risk_level"], frame[cat])
    chi2, p, _, _ = chi2_contingency(table)
    return float(chi2), float(p)


def bootstrap_odds(frame: pd.DataFrame, n_boot: int = 300, seed: int = 11) -> pd.DataFrame:
    """Odds of carrying any lexicon term, medium/high vs low, with and without length."""
    y = frame["any term"].to_numpy()
    base = np.column_stack([
        (frame["risk_level"] == "medium").astype(float),
        (frame["risk_level"] == "high").astype(float),
    ])
    adjusted = np.column_stack([base, np.log1p(frame["text_length"].to_numpy())])
    rng = np.random.default_rng(seed)

    draws = {"unadjusted": [], "length-adjusted": []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        for name, matrix in (("unadjusted", base), ("length-adjusted", adjusted)):
            model = LogisticRegression(max_iter=1000).fit(matrix[idx], y[idx])
            draws[name].append(np.exp(model.coef_[0][:2]))

    rows = []
    for name, values in draws.items():
        values = np.asarray(values)
        for level, col in (("medium", 0), ("high", 1)):
            lo, med, hi = np.percentile(values[:, col], [2.5, 50, 97.5])
            rows.append({"model": name, "level": level, "or": med, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)


def per_intent(frame: pd.DataFrame) -> pd.DataFrame:
    table = (frame.groupby(["intent", "risk_level"], observed=True)["any term"]
             .agg(["mean", "size"]).reset_index())
    return table.rename(columns={"mean": "share", "size": "n_messages"})


def sensitivity(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, words in SENSITIVITY.items():
        flag = frame["text"].str.contains(_pattern(words), regex=True)
        for level in ("low", "medium", "high"):
            sel = frame["risk_level"] == level
            rows.append({"lexicon": name, "risk_level": level,
                         "share": float(flag[sel].mean())})
    return pd.DataFrame(rows)


def draw(by_risk, by_type, odds, pvals, intent_df, sens):
    fig = plt.figure(figsize=(18.5, 13.4))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.15], wspace=0.24, hspace=0.40,
                            left=0.055, right=0.985, top=0.905, bottom=0.065)
    ax_a = fig.add_subplot(grid[0, 0])
    ax_c = fig.add_subplot(grid[0, 1])
    ax_b = fig.add_subplot(grid[1, 0])
    ax_d = fig.add_subplot(grid[1, 1])

    levels = ["low", "medium", "high"]
    cats = list(LEXICON) + ["any term"]
    width = 0.2

    # ---- a. share by risk level ------------------------------------------------
    for i, cat in enumerate(cats):
        sub = by_risk[by_risk["group"] == cat].set_index("risk_level").loc[levels]
        pos = np.arange(len(levels)) + (i - (len(cats) - 1) / 2) * width
        color = CAT_COLOR.get(cat, "#34495e")
        err = np.vstack([sub["share"] - sub["lo"], sub["hi"] - sub["share"]])
        ax_a.bar(pos, sub["share"], width=width * 0.92, color=color, label=cat,
                 yerr=err, capsize=2.0, error_kw={"lw": 0.8, "alpha": 0.8})
        for xpos, value in zip(pos, sub["share"]):
            ax_a.text(xpos, value + 0.008, f"{value:.1%}", ha="center", va="bottom",
                      rotation=90, fontsize=6.6, color="#333333")
    ax_a.set_xticks(range(len(levels)), [RISK_LABEL[l] for l in levels])
    ax_a.set_xlabel("intent risk level (number of intents in brackets)", fontsize=9)
    ax_a.set_ylabel("share of messages containing a term")
    ax_a.set_ylim(0, max(by_risk["hi"].max() * 1.22, 0.35))
    ax_a.legend(frameon=False, fontsize=8.2, loc="upper left")
    ax_a.grid(axis="y", alpha=0.25, lw=0.6)
    ax_a.set_axisbelow(True)
    p_text = "chi-square across risk levels:  " + "   ".join(
        f"{cat.split(' ')[0]} p={pvals[cat]:.1e}" for cat in LEXICON)
    ax_a.set_title("a. Urgency terms by risk level (Wilson 95% CI)\n" + p_text,
                   loc="left", fontsize=10)

    # ---- c. forest plot --------------------------------------------------------
    ypos = {"high": 1.0, "medium": 0.0}
    offset = {"length-adjusted": 0.14, "unadjusted": -0.14}
    color = {"length-adjusted": "#1f4e79", "unadjusted": "#9aa5ac"}
    for level in ("high", "medium"):
        for model in ("length-adjusted", "unadjusted"):
            row = odds[(odds["level"] == level) & (odds["model"] == model)].iloc[0]
            y = ypos[level] + offset[model]
            ax_c.errorbar(row["or"], y,
                          xerr=[[row["or"] - row["lo"]], [row["hi"] - row["or"]]],
                          fmt="o", ms=7, color=color[model], capsize=3.5, lw=1.6,
                          label=model if level == "high" else None)
            ax_c.text(row["hi"] * 1.07, y,
                      f"{row['or']:.2f}  [{row['lo']:.2f}-{row['hi']:.2f}]",
                      va="center", fontsize=8.2, color=color[model])
    ax_c.axvline(1.0, color="#555555", lw=0.9, ls="--")
    ax_c.set_yticks([1.0, 0.0], ["high vs low", "medium vs low"])
    ax_c.set_xscale("log")
    ax_c.set_xlim(0.85, 9.5)
    ax_c.xaxis.set_major_locator(FixedLocator([1, 1.5, 2, 3, 4, 6, 8]))
    ax_c.set_xticklabels(["1", "1.5", "2", "3", "4", "6", "8"])
    ax_c.xaxis.set_minor_formatter(NullFormatter())
    ax_c.text(0.02, 0.93, "for reference: log-length alone OR 8.2", transform=ax_c.transAxes,
              fontsize=7.8, color="#666666")
    ax_c.set_xlabel("odds ratio of carrying any urgency term (log scale)")
    ax_c.set_title("c. Does message length explain it?\n"
                   "same odds with (dark) and without (grey) log length in the model",
                   loc="left", fontsize=10)
    ax_c.grid(axis="x", alpha=0.25, lw=0.6)
    ax_c.set_axisbelow(True)
    ax_c.legend(frameon=False, fontsize=8.4, loc="lower right")

    # ---- b. share by fraud type ------------------------------------------------
    types = by_type["fraud_type"].unique().tolist()
    y = np.arange(len(types))
    for i, cat in enumerate(cats):
        sub = by_type[by_type["group"] == cat].set_index("fraud_type").loc[types]
        pos = y + (i - (len(cats) - 1) / 2) * width
        col = CAT_COLOR.get(cat, "#34495e")
        err = np.clip(np.vstack([sub["share"] - sub["lo"], sub["hi"] - sub["share"]]), 0, None)
        ax_b.barh(pos, sub["share"], height=width * 0.92, color=col, label=cat,
                  xerr=err, capsize=1.6, error_kw={"lw": 0.7, "alpha": 0.8})
        if cat in ("any term", "incident / threat"):
            for ypos_, value in zip(pos, sub["share"]):
                if value >= 0.02:
                    ax_b.text(value + 0.008, ypos_, f"{value:.0%}", va="center",
                              fontsize=6.8, color="#333333")
    ax_b.set_yticks(y, [t.replace("_", " ") for t in types], fontsize=8.8)
    ax_b.invert_yaxis()
    ax_b.set_xlim(0, max(by_type["hi"].max() * 1.16, 0.1))
    ax_b.set_xlabel("share of messages containing a term", fontsize=9)
    ax_b.set_title("b. The same terms by fraud type ('ordinary' = all 55 low-risk intents)",
                   loc="left", fontsize=10)
    ax_b.grid(axis="x", alpha=0.25, lw=0.6)
    ax_b.set_axisbelow(True)

    # ---- d. per-intent spread --------------------------------------------------
    rng = np.random.default_rng(3)
    for i, level in enumerate(levels):
        sub = intent_df[intent_df["risk_level"] == level]
        jitter = rng.uniform(-0.22, 0.22, len(sub))
        ax_d.scatter(np.full(len(sub), i) + jitter, sub["share"],
                     s=18 + sub["n_messages"] / 8, color=RISK_COLOR[level],
                     alpha=0.75, edgecolor="white", linewidth=0.5, zorder=3)
        med = sub["share"].median()
        ax_d.hlines(med, i - 0.32, i + 0.32, color="#333333", lw=2.0, zorder=4)
        ax_d.text(i + 0.35, med, f"median {med:.0%}", va="center", fontsize=8,
                  color="#333333")
    ax_d.set_xticks(range(len(levels)), [RISK_LABEL[l] for l in levels])
    ax_d.set_xlim(-0.6, 2.75)
    ax_d.set_ylim(0, 0.95)
    ax_d.set_ylabel("share of that intent's messages with a term", fontsize=9)
    ax_d.set_title("d. Each dot is one intent: the spread inside a risk level dwarfs the gap "
                   "between levels\n(dot area = messages in that intent; bar = median)",
                   loc="left", fontsize=10)
    ax_d.grid(axis="y", alpha=0.25, lw=0.6)
    ax_d.set_axisbelow(True)

    highlights = {
        "compromised_card": (1.80, 0.90, "right"),
        "lost_or_stolen_card": (1.80, 0.78, "right"),
        "why_verify_identity": (1.80, 0.10, "right"),
        "terminate_account": (0.22, 0.40, "left"),
    }
    for name, (tx, ty, ha) in highlights.items():
        row = intent_df[intent_df["intent"] == name].iloc[0]
        i = levels.index(row["risk_level"])
        ax_d.annotate(name, (i, row["share"]), xytext=(tx, ty),
                      fontsize=7.6, ha=ha, va="center", color="#333333",
                      arrowprops=dict(arrowstyle="-", lw=0.6, color="#999999",
                                      shrinkA=2, shrinkB=4))

    fig.suptitle(
        "B3  Urgency words track risk only through incident vocabulary\n"
        "time pressure (immediately / urgent) stays flat at 4-6% across risk levels; "
        "incident words (someone / stolen) go 0.8% to 18.3%, mostly from card compromise",
        x=0.012, y=0.975, ha="left", fontsize=13.5, fontweight="bold",
    )
    fig.text(0.5, 0.012,
             "lexicon sensitivity (share carrying a term, low / medium / high): "
             + "    ".join(
                 f"{name}: " + " / ".join(
                     f"{sens[(sens.lexicon == name) & (sens.risk_level == l)]['share'].iloc[0]:.1%}"
                     for l in levels)
                 for name in SENSITIVITY),
             ha="center", fontsize=8.2, color="#444444")
    return fig


def main() -> None:
    apply_style()
    clean, dim = load_data()
    frame = build_flags(clean, dim)

    by_risk = share_table(frame, "risk_level")
    by_risk["risk_level"] = pd.Categorical(by_risk["risk_level"],
                                           categories=["low", "medium", "high"], ordered=True)
    by_risk = by_risk.sort_values(["group", "risk_level"])

    fraud = frame.dropna(subset=["fraud_type"]).copy()
    ordinary = frame[frame["risk_level"] == "low"].copy()
    ordinary["fraud_type"] = "ordinary"
    by_type = share_table(pd.concat([fraud, ordinary], ignore_index=True), "fraud_type")
    preferred = ["ordinary", "virtual_card_abuse", "identity_kyc_bypass", "account_takeover",
                 "app_fraud_money_mule", "refund_chargeback_abuse", "card_compromise"]
    by_type["fraud_type"] = pd.Categorical(by_type["fraud_type"], categories=preferred,
                                           ordered=True)
    by_type = by_type.sort_values(["group", "fraud_type"])

    pvals = {cat: chi_square(frame, cat)[1] for cat in LEXICON}
    odds = bootstrap_odds(frame)
    intent_df = per_intent(frame)
    sens = sensitivity(frame)

    fig = draw(by_risk, by_type, odds, pvals, intent_df, sens)
    save_figure(fig, "B_urgency_by_risk.png")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    by_risk.to_csv(OUT_DIR / "b3_shares_by_risk.csv", index=False)
    by_type.to_csv(OUT_DIR / "b3_shares_by_fraud_type.csv", index=False)
    odds.to_csv(OUT_DIR / "b3_odds_ratios.csv", index=False)
    intent_df.to_csv(OUT_DIR / "b3_per_intent.csv", index=False)
    sens.to_csv(OUT_DIR / "b3_lexicon_sensitivity.csv", index=False)
    (OUT_DIR / "b3_chi_square.json").write_text(json.dumps(pvals, indent=2))

    print("share by risk level:")
    print(by_risk.pivot(index="risk_level", columns="group", values="share").round(3).to_string())
    print("\nchi-square p:", {k: f"{v:.2e}" for k, v in pvals.items()})
    print("\nodds ratios:\n", odds.round(3).to_string(index=False))
    print("\nlexicon sensitivity:\n",
          sens.pivot(index="lexicon", columns="risk_level", values="share").round(3).to_string())
    spread = intent_df.groupby("risk_level")["share"].agg(["min", "median", "max"]).round(3)
    print("\nper-intent spread:\n", spread.to_string())


main()

In [ ]:
from IPython.display import Image, display
from _common import FIG_DIR, OUT_DIR
import pandas as pd

display(Image(filename=str(FIG_DIR / "B_urgency_by_risk.png")))
pd.read_csv(OUT_DIR / "b3_lexicon_sensitivity.csv").pivot(
    index="lexicon", columns="risk_level", values="share").round(3)

**Interpretation (3 sentences).** Urgency words do not track risk the way intuition suggests:
time-pressure terms sit at 4.1% / 6.1% / 4.5% across low, medium and high risk, and the whole
gradient comes from incident words (`someone`, `stolen`), which rise from 0.8% to 18.3% and
are concentrated almost entirely in card compromise. Two fraud types are *calmer* than
ordinary customers — virtual-card abuse 5.8% and KYC bypass 4.3% against 9.9% — so a "fraud
sounds urgent" heuristic would miss precisely the quiet frauds a bank most wants to catch.
Even inside the high-risk group the signal is uneven (1.9% for `why_verify_identity` against
85.7% for `compromised_card`), and it survives the length control (high-vs-low odds ratio
3.35 → 4.80 once log length is added), which is why the fraud model has to lean on behaviour
rather than tone.

**Caveats.** The lexicon is fixed by the brief, not fitted; the sensitivity table shows the
high-above-low ordering holds for brief-only (3.3/6.9/20.9), current (9.9/15.1/27.0) and
broad (16.7/22.6/38.7) lists, and collapses only when the list is restricted to strict
urgency words (0.7/2.8/1.7) — which is itself the point.

## Summary — what part B says about text

| figure | finding | implication for the fraud model |
| --- | --- | --- |
| **B1** | 89% intent accuracy, but 6.4% of fraud-typed messages are routed out of the fraud workflow; virtual-card abuse leaks 12.5% | text-only intent routing loses fraud cases before any fraud model runs |
| **B2** | every fraud type has its own vocabulary, but a standalone keyword rule has 77% recall and only 29% precision | keywords are triage aids, not detectors |
| **B3** | urgency tone fires only for card compromise; identity, KYC and virtual-card abuse read calmly | tone is a weak, type-specific feature; behaviour must carry the rest |

**Limits of this part.** Everything here is measured on Banking77 text only: 13,083 messages,
77 intents, ~170 messages per intent and 3,080 test rows, so per-pair counts of 3–9 messages
are noisy (hence the bootstrap and Wilson intervals). `risk_level` and `fraud_type` come from
the group's own `config/intent_taxonomy.yaml`, i.e. they are domain labels, not ground truth
from the bank. No simulated stream is used in this part.

**Where this hands off.** The 6.4% leakage and the 29% keyword precision are the quantitative
case for the ML section's comparison of a text-only detector against one that adds user
behaviour — the gap between them is the value of the C/D figures.